# Tinker Tutorial: Fine-Tuning LLMs with SFT and RL


1. What Tinker is and why we use it
2. How to do **Supervised Fine-Tuning (SFT)** on a language model
3. How to do **Reinforcement Learning (RL)** to improve reasoning

This tutorial is adapted from the tinker-cookbook recipes. For the full implementations, see:
- SFT: [sl_loop.py](https://github.com/thinking-machines-lab/tinker-cookbook/blob/main/tinker_cookbook/recipes/sl_loop.py)
- RL: [rl_loop.py](https://github.com/thinking-machines-lab/tinker-cookbook/blob/main/tinker_cookbook/recipes/rl_loop.py)

> **Note:** This tutorial is a simplified version for teaching purposes. It omits features like learning rate scheduling, checkpointing, logging, and advanced configuration. For full, production-ready implementations, refer to the [tinker-cookbook](https://github.com/thinking-machines-lab/tinker-cookbook), in particular the [recipes/](https://github.com/thinking-machines-lab/tinker-cookbook/tree/main/tinker_cookbook/recipes) directory. They also contain task training examples, which can be good starting point for the final project.


---

## Part 0: Background — What is Fine-Tuning?

Large Language Models (LLMs) like Llama and Qwen are **pre-trained** on massive text corpora. They learn general language patterns, but they don't know how to follow instructions, solve math problems, or behave safely out of the box.

**Fine-tuning** adapts a pre-trained model for a specific task. There are two main approaches:

### Supervised Fine-Tuning (SFT)
- You provide **(input, desired output)** pairs
- The model learns to produce outputs similar to your examples
- Think of it as "teaching by example"
- Example: showing the model `(question, answer)` pairs for a Q&A task

### Reinforcement Learning (RL)
- The model **generates** outputs, and a **reward function** scores them
- The model learns to produce outputs that get higher scores
- Think of it as "learning from trial and error"
- Example: the model attempts math problems, and we reward correct answers

### What is Tinker?

[Tinker](https://tinker-docs.thinkingmachines.ai/) is a **distributed LLM fine-tuning API** by Thinking Machines Lab. It handles all the complex GPU infrastructure for you — you just write the training logic in Python.

Key features:
- **LoRA** (Low-Rank Adaptation): trains only a small fraction of model parameters, making fine-tuning fast and memory-efficient
- **Managed infrastructure**: no need to set up GPU servers yourself
- **Simple API**: `forward_backward()` → `optim_step()` → done!

### What is LoRA?

Instead of updating all billions of parameters, LoRA adds small trainable matrices ("adapters") to the model. This means:
- **Much less memory** needed
- **Much faster** training
- Results are surprisingly close to full fine-tuning

The `rank` parameter controls adapter size — higher rank = more capacity but slower.

---
## Part 1: Setup

In [ ]:
# Install Tinker SDK (run once)
!pip install tinker -q

In [ ]:
# Set your API key (get one from https://tinker-console.thinkingmachines.ai)
import os
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- Replace with your key

In [26]:
import tinker
from tinker import types
import numpy as np

# Verify connection and see available models
service_client = tinker.ServiceClient()
capabilities = service_client.get_server_capabilities()

print("Available models:")
for m in capabilities.supported_models:
    print(f"  - {m.model_name}")

Your Tinker SDK version is outdated. Please upgrade to the latest version.


Available models:
  - deepseek-ai/DeepSeek-V3.1
  - deepseek-ai/DeepSeek-V3.1-Base
  - moonshotai/Kimi-K2-Thinking
  - moonshotai/Kimi-K2.5
  - meta-llama/Llama-3.1-70B
  - meta-llama/Llama-3.1-8B
  - meta-llama/Llama-3.1-8B-Instruct
  - meta-llama/Llama-3.2-1B
  - meta-llama/Llama-3.2-3B
  - meta-llama/Llama-3.3-70B-Instruct
  - nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  - nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
  - Qwen/Qwen3-235B-A22B-Instruct-2507
  - Qwen/Qwen3-30B-A3B
  - Qwen/Qwen3-30B-A3B-Base
  - Qwen/Qwen3-30B-A3B-Instruct-2507
  - Qwen/Qwen3-32B
  - Qwen/Qwen3-4B-Instruct-2507
  - Qwen/Qwen3-8B
  - Qwen/Qwen3-8B-Base
  - Qwen/Qwen3-VL-235B-A22B-Instruct
  - Qwen/Qwen3-VL-30B-A3B-Instruct
  - Qwen/Qwen3.5-27B
  - Qwen/Qwen3.5-35B-A3B
  - Qwen/Qwen3.5-397B-A17B
  - Qwen/Qwen3.5-4B
  - openai/gpt-oss-120b
  - openai/gpt-oss-20b


---
## Part 2: Supervised Fine-Tuning (SFT)

We'll fine-tune a model to translate English to Pig Latin — a simple task that clearly demonstrates the SFT workflow.

**Pig Latin rules:** move the first consonant cluster to the end and add "ay".
- "hello" → "ellohay"
- "world" → "orldway"
- "apple" → "appleay" (starts with vowel, just add "ay")

### Step 1: Prepare Training Data

SFT needs **(prompt, completion)** pairs. We need to:
1. Tokenize both prompt and completion
2. Set **loss weights**: 0 for prompt tokens (don't learn to copy the question), 1 for completion tokens (learn to produce the answer)

This is packaged into a `Datum` object.

In [27]:
# Choose a base model
BASE_MODEL = "meta-llama/Llama-3.1-8B"

# Create a training client with LoRA
training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=32,  # LoRA rank — 32 is a good default
)

# Get the tokenizer (converts text ↔ token IDs)
tokenizer = training_client.get_tokenizer()

print(f"Model: {BASE_MODEL}")
print(f"LoRA rank: 32")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Model: meta-llama/Llama-3.1-8B
LoRA rank: 32
Tokenizer vocab size: 128000


In [28]:
# Our training dataset: English → Pig Latin pairs
train_examples = [
    {"input": "hello world", "output": "ellohay orldway"},
    {"input": "good morning", "output": "oodgay orningmay"},
    {"input": "machine learning", "output": "achinemay earninglay"},
    {"input": "natural language", "output": "aturalnay anguagelay"},
    {"input": "deep neural network", "output": "eepday euralnay etworknay"},
    {"input": "fine tuning models", "output": "inefay uningtay odelsmay"},
    {"input": "banana split", "output": "ananabay itsplay"},
    {"input": "train test split", "output": "aintray esttay itsplay"},
    {"input": "gradient descent", "output": "adientgray escentday"},
    {"input": "back propagation", "output": "ackbay opagationpray"},
    {"input": "transfer learning", "output": "ansfertray earninglay"},
    {"input": "attention mechanism", "output": "attentionay echanismmay"},
    {"input": "token embedding", "output": "okentay embeddingay"},
    {"input": "batch normalization", "output": "atchbay ormalizationnay"},
    {"input": "cross entropy loss", "output": "osscray entropyay osslay"},
    {"input": "language model", "output": "anguagelay odelmay"},
]

print(f"Training examples: {len(train_examples)}")
print(f"Example: '{train_examples[0]['input']}' → '{train_examples[0]['output']}'")

Training examples: 16
Example: 'hello world' → 'ellohay orldway'


In [29]:
def process_example(example: dict, tokenizer) -> types.Datum:
    """
    Convert a training example into a Datum for Tinker.

    Key idea: we shift tokens so that at each position, the model
    predicts the NEXT token. This is standard for language model training.

    Tokens:      [A, B, C, D, E]
    Input:       [A, B, C, D]      ← model sees these
    Target:      [B, C, D, E]      ← model predicts these
    Weights:     [0, 0, 1, 1]      ← only train on completion tokens
    """
    prompt = f"English: {example['input']}\nPig Latin:"
    completion = f" {example['output']}\n\n"

    # Tokenize prompt and completion separately
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    completion_tokens = tokenizer.encode(completion, add_special_tokens=False)

    # Combine tokens
    all_tokens = prompt_tokens + completion_tokens

    # Loss weights: 0 for prompt, 1 for completion
    weights = [0] * len(prompt_tokens) + [1] * len(completion_tokens)

    # Shift for next-token prediction
    input_tokens = all_tokens[:-1]
    target_tokens = all_tokens[1:]
    weights = weights[1:]  # align weights with targets

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs=dict(
            weights=weights,
            target_tokens=target_tokens,
        ),
    )

# Process all examples
processed_data = [process_example(ex, tokenizer) for ex in train_examples]

# Let's inspect one example
ex = train_examples[0]
prompt_tokens = tokenizer.encode(f"English: {ex['input']}\nPig Latin:", add_special_tokens=True)
completion_tokens = tokenizer.encode(f" {ex['output']}\n\n", add_special_tokens=False)
print(f"Prompt: 'English: {ex['input']}\\nPig Latin:'")
print(f"Completion: ' {ex['output']}\\n\\n'")
print(f"Prompt tokens: {len(prompt_tokens)}, Completion tokens: {len(completion_tokens)}")
print(f"Total tokens per example: {len(prompt_tokens) + len(completion_tokens)}")
print(f"\nTotal Datum objects: {len(processed_data)}")

Prompt: 'English: hello world\nPig Latin:'
Completion: ' ellohay orldway\n\n'
Prompt tokens: 10, Completion tokens: 6
Total tokens per example: 16

Total Datum objects: 16


### Step 2: Training Loop

The SFT training loop is simple:
1. **`forward_backward()`** — compute the loss and gradients on a batch
2. **`optim_step()`** — update the model weights using Adam optimizer
3. Repeat!

We use `"cross_entropy"` as the loss function — the standard loss for language modeling.

In [30]:
from tinker_cookbook.supervised.common import compute_mean_nll

NUM_STEPS = 6
LEARNING_RATE = 1e-4  # LoRA uses higher LR than full fine-tuning

print(f"Training for {NUM_STEPS} steps...\n")

for step in range(NUM_STEPS):
    # Step 1: Forward pass (compute loss) + Backward pass (compute gradients)
    fwdbwd_future = training_client.forward_backward(processed_data, "cross_entropy")

    # Step 2: Update weights with Adam optimizer
    optim_future = training_client.optim_step(
        types.AdamParams(learning_rate=LEARNING_RATE)
    )

    # Wait for results and compute loss
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    # Calculate average loss (weighted negative log-likelihood)
    logprobs = [x["logprobs"] for x in fwdbwd_result.loss_fn_outputs]
    weights = [d.loss_fn_inputs["weights"] for d in processed_data]
    loss = compute_mean_nll(logprobs, weights)

    print(f"Step {step + 1}/{NUM_STEPS} | Loss: {loss:.4f}")

print("\nTraining complete! Loss should be decreasing.")

Training for 6 steps...

Step 1/6 | Loss: 3.5643
Step 2/6 | Loss: 2.7235
Step 3/6 | Loss: 1.4601
Step 4/6 | Loss: 0.8259
Step 5/6 | Loss: 0.3949
Step 6/6 | Loss: 0.1849

Training complete! Loss should be decreasing.


### Step 3: Test the Fine-Tuned Model

Now let's see if our model learned Pig Latin! We'll:
1. Save the LoRA weights and create a **sampling client** (for inference)
2. Generate translations for new inputs

In [31]:
# Save weights and create a sampling (inference) client
sampling_client = training_client.save_weights_and_get_sampling_client(
    name="pig-latin-model"
)

print("Model saved! Ready for inference.")

Model saved! Ready for inference.


In [32]:
def translate(text: str) -> str:
    """Use our fine-tuned model to translate English → Pig Latin."""
    prompt = f"English: {text}\nPig Latin:"
    prompt_tokens = types.ModelInput.from_ints(
        tokenizer.encode(prompt, add_special_tokens=True)
    )

    result = sampling_client.sample(
        prompt=prompt_tokens,
        sampling_params=types.SamplingParams(
            max_tokens=50,
            temperature=0.0,  # greedy decoding (deterministic)
            stop=["\n"],     # stop at newline
        ),
        num_samples=1,
    ).result()

    return tokenizer.decode(result.sequences[0].tokens).strip()

# Test on training examples (should work well)
print("=== On training examples ===")
for ex in train_examples[:4]:
    pred = translate(ex["input"])
    status = "correct" if pred == ex["output"] else "wrong"
    print(f"  '{ex['input']}' → '{pred}' (expected: '{ex['output']}') [{status}]")

# Test on NEW examples (generalization)
print("\n=== On new examples (generalization) ===")
test_inputs = ["coffee break", "python code", "student teacher"]
for text in test_inputs:
    pred = translate(text)
    print(f"  '{text}' → '{pred}'")

=== On training examples ===
  'hello world' → 'ellohay orldway' (expected: 'ellohay orldway') [correct]
  'good morning' → 'oodgay orningmay' (expected: 'oodgay orningmay') [correct]
  'machine learning' → 'achinemay earninglay' (expected: 'achinemay earninglay') [correct]
  'natural language' → 'aturalnay anguagelay' (expected: 'aturalnay anguagelay') [correct]

=== On new examples (generalization) ===
  'coffee break' → 'offeyay eakbay'
  'python code' → 'ythonpay odecay'
  'student teacher' → 'atudentstay eachertay'


### SFT Recap

What we did:
1. **Created a training client** with LoRA (`create_lora_training_client`)
2. **Prepared data** as `Datum` objects with loss weights (0 for prompt, 1 for completion)
3. **Trained** with `forward_backward()` + `optim_step()` loop
4. **Deployed** with `save_weights_and_get_sampling_client()`
5. **Tested** with `sample()`

**Key takeaway:** SFT is great when you have high-quality example outputs. The model learns to imitate your examples.

---
## [optional] Part 3: Reinforcement Learning (RL)

### When to use RL instead of SFT?

| | SFT | RL |
|---|---|---|
| **Data needed** | (input, output) pairs | inputs + reward function |
| **Best for** | Tasks with clear "correct" outputs | Tasks where outputs can be scored |
| **Learning style** | Imitation | Trial and error |
| **Example** | Translation, summarization | Math reasoning, code generation |

RL is powerful when:
- You can **automatically verify** if an output is correct
- There are many valid outputs (not just one "right answer")
- You want the model to develop its own reasoning strategies

### Our Task: Grade School Math (GSM8K)

We'll use a subset of [GSM8K](https://arxiv.org/abs/2110.14168) — a dataset of grade-school math word problems. The reward is simple: **1 if the answer is correct, 0 otherwise**.

In [33]:
# Sample GSM8K problems for our tutorial
gsm8k_problems = [
    {
        "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
        "answer": 18,
    },
    {
        "question": "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
        "answer": 3,
    },
    {
        "question": "Josh decides to try flipping a house. He buys a house for $80,000 and puts $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?",
        "answer": 70000,
    },
    {
        "question": "James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?",
        "answer": 540,
    },
    {
        "question": "Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy. She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed. In the afternoon, she gives her chickens another 25 cups of feed. If Wendi has 20 chickens, how many cups of feed does she need to give her chickens in the final meal of the day?",
        "answer": 20,
    },
    {
        "question": "Kylar went to the store to get his 2 gallons of whole milk but they didn't have any. So he bought 2 gallons of 2% milk for $3.50 a gallon and 2 gallons of whole milk substitute for $3.50 a gallon. He paid with a $20 bill. How much change did he get?",
        "answer": 6,
    },
    {
        "question": "Toulouse has twice as many sheep as Charleston. Charleston has 4 times as many sheep as Seattle. How many sheep do Toulouse, Charleston, and Seattle have together if Seattle has 20 sheep?",
        "answer": 260,
    },
    {
        "question": "Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load(), in minutes, does it take to download the file?",
        "answer": 160,
    },
]

print(f"Number of problems: {len(gsm8k_problems)}")
print(f"\nExample problem:")
print(f"  Q: {gsm8k_problems[0]['question'][:80]}...")
print(f"  A: {gsm8k_problems[0]['answer']}")

Number of problems: 8

Example problem:
  Q: Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning an...
  A: 18


### Step 1: Define the Reward Function

In RL, the **reward function** tells the model how well it did. For math:
- Extract the numerical answer from the model's response
- Compare with the ground truth
- Reward = 1 if correct, 0 if wrong

In [34]:
import re

def extract_answer(text: str):
    """
    Extract the final numerical answer from the model's response.
    Looks for patterns like:
      'The answer is 42'
      '#### 42'
      'Answer: 42'
    """
    # Try common answer formats
    patterns = [
        r"####\s*(-?[\d,]+)",           # GSM8K format: #### 42
        r"answer is\s*(-?[\d,]+)",       # "the answer is 42"
        r"Answer:\s*(-?[\d,]+)",         # "Answer: 42"
        r"=\s*(-?[\d,]+)\s*$",           # "= 42" at end
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return int(match.group(1).replace(",", ""))

    # Fallback: last number in the text
    numbers = re.findall(r"-?\d+", text)
    if numbers:
        return int(numbers[-1])
    return None


def compute_reward(model_output: str, correct_answer: int) -> float:
    """Return 1.0 if the model's answer matches, 0.0 otherwise."""
    predicted = extract_answer(model_output)
    return 1.0 if predicted == correct_answer else 0.0


# Quick test
print("Reward function tests:")
print(f"  'The answer is 18' → reward = {compute_reward('The answer is 18', 18)}")
print(f"  '#### 42'          → reward = {compute_reward('#### 42', 42)}")
print(f"  'I think it is 5'  → reward = {compute_reward('I think it is 5', 10)}")

Reward function tests:
  'The answer is 18' → reward = 1.0
  '#### 42'          → reward = 1.0
  'I think it is 5'  → reward = 0.0


### Step 2: The RL Training Loop

RL training follows this cycle:

```
For each problem:
  1. SAMPLE: Generate multiple responses ("rollouts")
  2. REWARD: Score each response with the reward function
  3. LEARN:  Compute advantages → forward_backward → optim_step
```

**Why multiple samples?** By generating many responses per problem, we can compare good vs bad attempts. The model learns to produce more outputs like the high-reward ones (this is the REINFORCE algorithm).

**Advantage** = how much better this response was compared to the average for this problem. Positive advantage → reinforce this behavior. Negative → discourage it.

In [35]:
# Create a fresh training client for RL
rl_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=32,
)
rl_tokenizer = rl_training_client.get_tokenizer()

# Get a sampling client from the current (untrained) weights
rl_sampling_client = rl_training_client.save_weights_and_get_sampling_client(
    name="rl-math-sampler"
)

print("RL training client ready!")

RL training client ready!


In [ ]:
# RL hyperparameters
GROUP_SIZE = 4       # number of responses per problem
MAX_TOKENS = 256     # max completion length
RL_LEARNING_RATE = 4e-5
NUM_ITERATIONS = 3   # keep small for demo


def build_math_prompt(question: str) -> str:
    """Format a math problem as a prompt."""
    return (
        f"Solve this math problem step by step. "
        f"End with 'The answer is <number>'.\n\n"
        f"Problem: {question}\n\n"
        f"Solution:"
    )


print("Example prompt:")
print(build_math_prompt(gsm8k_problems[0]["question"])[:200] + "...")

Example prompt:
Solve this math problem step by step. End with 'The answer is <number>'.

Problem: Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every...


In [37]:
import torch
from tinker.types.tensor_data import TensorData

print(f"Starting RL training: {NUM_ITERATIONS} iterations\n")
print(f"Each iteration: {len(gsm8k_problems)} problems × {GROUP_SIZE} samples = "
      f"{len(gsm8k_problems) * GROUP_SIZE} rollouts\n")

for iteration in range(NUM_ITERATIONS):
    datums_D = []
    total_reward = 0
    total_samples = 0

    # Phase 1: Sample responses and compute rewards
    for problem in gsm8k_problems:
        prompt_text = build_math_prompt(problem["question"])
        prompt_tokens = rl_tokenizer.encode(prompt_text, add_special_tokens=True)
        prompt_input = types.ModelInput.from_ints(prompt_tokens)

        # Generate GROUP_SIZE responses for this problem
        sample_result = rl_sampling_client.sample(
            prompt=prompt_input,
            sampling_params=types.SamplingParams(
                max_tokens=MAX_TOKENS,
                temperature=0.7,  # some randomness for exploration
            ),
            num_samples=GROUP_SIZE,
        ).result()

        # Collect rewards, tokens, and logprobs for each response
        rewards_G = []
        sampled_tokens_G_T = []
        logprobs_G_T = []
        for seq in sample_result.sequences:
            sampled_tokens_G_T.append(seq.tokens)
            logprobs_G_T.append(seq.logprobs)

            completion_text = rl_tokenizer.decode(seq.tokens)
            reward = compute_reward(completion_text, problem["answer"])
            rewards_G.append(reward)

        # Compute advantages (reward - mean_reward for this problem)
        mean_reward = sum(rewards_G) / len(rewards_G)
        advantages_G = [r - mean_reward for r in rewards_G]

        total_reward += sum(rewards_G)
        total_samples += len(rewards_G)

        # Skip if all advantages are zero (no learning signal)
        if all(a == 0.0 for a in advantages_G):
            continue

        # Phase 2: Build training datums from rollouts
        # Following the cookbook pattern (rl_loop.py):
        #   - Use importance_sampling loss (not cross_entropy)
        #   - Include logprobs and advantages in loss_fn_inputs
        #   - Pad with zeros for the prompt portion
        ob_len = prompt_input.length - 1
        for sampled_tokens, logprobs, advantage in zip(
            sampled_tokens_G_T, logprobs_G_T, advantages_G
        ):
            model_input = prompt_input.append(
                types.EncodedTextChunk(tokens=sampled_tokens[:-1])
            )
            target_tokens = [0] * ob_len + list(sampled_tokens)
            padded_logprobs = [0.0] * ob_len + list(logprobs)
            padded_advantages = [0.0] * ob_len + [advantage] * (model_input.length - ob_len)

            datum = types.Datum(
                model_input=model_input,
                loss_fn_inputs={
                    "target_tokens": TensorData.from_torch(torch.tensor(target_tokens)),
                    "logprobs": TensorData.from_torch(torch.tensor(padded_logprobs)),
                    "advantages": TensorData.from_torch(torch.tensor(padded_advantages)),
                },
            )
            datums_D.append(datum)

    accuracy = total_reward / total_samples

    # Phase 3: Update model weights
    if datums_D:
        fwdbwd_future = rl_training_client.forward_backward(datums_D, "importance_sampling")
        optim_future = rl_training_client.optim_step(
            types.AdamParams(learning_rate=RL_LEARNING_RATE)
        )
        fwdbwd_future.result()
        optim_future.result()

        # Update sampling client with new weights
        rl_sampling_client = rl_training_client.save_weights_and_get_sampling_client(
            name=f"rl-math-iter-{iteration}"
        )

    print(f"Iteration {iteration + 1}/{NUM_ITERATIONS} | "
          f"Accuracy: {accuracy:.1%} | "
          f"Training samples: {len(datums_D)}")

print("\nRL training complete!")

Starting RL training: 3 iterations

Each iteration: 8 problems × 4 samples = 32 rollouts

Iteration 1/3 | Accuracy: 31.2% | Training samples: 24
Iteration 2/3 | Accuracy: 25.0% | Training samples: 24
Iteration 3/3 | Accuracy: 21.9% | Training samples: 16

RL training complete!


### Step 3: Evaluate the RL-Trained Model

In [38]:
print("=== Evaluating RL-trained model ===")
print("(Generating with temperature=0 for deterministic output)\n")

correct = 0
for problem in gsm8k_problems[:5]:  # test on first 5 problems
    prompt_text = build_math_prompt(problem["question"])
    prompt_input = types.ModelInput.from_ints(
        rl_tokenizer.encode(prompt_text, add_special_tokens=True)
    )

    result = rl_sampling_client.sample(
        prompt=prompt_input,
        sampling_params=types.SamplingParams(
            max_tokens=MAX_TOKENS,
            temperature=0.0,
        ),
        num_samples=1,
    ).result()

    output = rl_tokenizer.decode(result.sequences[0].tokens)
    predicted = extract_answer(output)
    is_correct = predicted == problem["answer"]
    correct += int(is_correct)

    print(f"Q: {problem['question'][:60]}...")
    print(f"  Model output (last 100 chars): ...{output[-100:]}")
    print(f"  Predicted: {predicted} | Correct: {problem['answer']} | {'CORRECT' if is_correct else 'WRONG'}")
    print()

print(f"Final accuracy: {correct}/5 = {correct/5:.0%}")

=== Evaluating RL-trained model ===
(Generating with temperature=0 for deterministic output)

Q: Janet's ducks lay 16 eggs per day. She eats three for breakf...
  Model output (last 100 chars): ... 16 - 3 - 4 = 9 eggs left. 9 x 2 = 18 dollars.<|end_of_text|>
  Predicted: 18 | Correct: 18 | CORRECT

Q: A robe takes 2 bolts of blue fiber and half that much white ...
  Model output (last 100 chars): ...he number of bolts of white fiber. Then 2x is the number of bolts of blue fiber. The total number of
  Predicted: 3 | Correct: 3 | CORRECT

Q: Josh decides to try flipping a house. He buys a house for $8...
  Model output (last 100 chars): ...50,000 in repairs. The profit is the difference between the selling price and the cost of the house.
  Predicted: 0 | Correct: 70000 | WRONG

Q: James decides to run 3 sprints 3 times a week. He runs 60 me...
  Model output (last 100 chars): ... 3 x 4 + 2 x 4 = 16 legs in total.

The answer is 16 legs.

Problem: A farmer has 3 cows and 2 pigs.
  Predic

### RL Recap

What we did:
1. **Defined a reward function** that checks if math answers are correct
2. **Sampled** multiple responses per problem (`sample()` with `num_samples > 1`)
3. **Computed advantages** (how much better/worse than average)
4. **Updated the model** to reinforce correct solutions and discourage wrong ones
5. **Iterated** — each round uses the improved model for sampling

**Key takeaway:** RL doesn't need example solutions — it discovers them through exploration. The reward function is what guides learning.

---
## Part 4: Saving and Resuming

Tinker lets you save checkpoints and resume training later.

In [39]:
# Save a checkpoint
save_result = rl_training_client.save_state("my-checkpoint").result()
checkpoint_path = save_result.path
print(f"Checkpoint saved at: {checkpoint_path}")

# Later, you can resume from this checkpoint:
# resumed_client = service_client.create_training_client_from_state(checkpoint_path)

Checkpoint saved at: tinker://454ee091-2934-51b5-b114-ba907f90847d:train:1/weights/my-checkpoint


---
## Summary

| Concept | SFT | RL |
|---|---|---|
| **Input** | (prompt, completion) pairs | prompts + reward function |
| **Data prep** | `Datum` with loss weights 0/1 | Sample → reward → advantage weights |
| **Training** | `forward_backward` + `optim_step` | Same, but on rollouts with advantages |
| **When to use** | Have gold-standard outputs | Can verify correctness automatically |

### Tinker API Cheat Sheet

```python
# Setup
service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(base_model=..., rank=32)
tokenizer = training_client.get_tokenizer()

# Training
training_client.forward_backward(datums, "cross_entropy")  # compute gradients
training_client.optim_step(types.AdamParams(learning_rate=...))  # update weights

# Inference
sampling_client = training_client.save_weights_and_get_sampling_client(name=...)
sampling_client.sample(prompt=..., sampling_params=..., num_samples=...)

# Checkpointing
training_client.save_state("checkpoint-name")
service_client.create_training_client_from_state(path)
```

### Additional Materials

- **Tinker Docs:** https://tinker-docs.thinkingmachines.ai/
- **Tinker Cookbook:** https://github.com/thinking-machines-lab/tinker-cookbook (more recipes: chat SL, preference learning, tool use, multi-agent RL)